# ARCHS4 saturation: GWAS trait recovery at K = 1728

Same question as the pathway-recovery report, but for traits: the query set is phenotypes (PhenomeXcan rapid GWAS, scored via phenoplier's GLS) rather than gene sets, and the denominator is the fixed trait catalog size. Both models are forced to the full K=1728 rank; CLAMPfull_bp and CLAMPbase are reported as separate panels below (rs1/seed1 excluded for both: SVD rank too low for K=1728).

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
    library(data.table)
    library(ggplot2)
    library(here)
})

traits <- fread(here(snakemake@input[["traits_long"]]))
FDR <- unique(traits$fdr)
stopifnot(length(FDR) == 1)

traits[, model := factor(model, levels = c("CLAMPfull_bp", "CLAMPbase"))]
traits[, fraction_label := factor(sprintf("%d%%", fraction),
                                   levels = sprintf("%d%%", sort(unique(fraction))))]
traits[, .N, by = .(model, fraction_label)]

## Trait recovery against the full compendium

In [ ]:
stars_for <- function(p) {
    ifelse(is.na(p), "n/a",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01, "**",
    ifelse(p < 0.05, "*", "ns"))))
}

ref_tests <- function(d, ref) {
    dd <- droplevels(d)
    if (!ref %in% levels(dd$fraction_label)) return(data.table())
    others <- intersect(c("25%", "50%"), levels(dd$fraction_label))
    if (!length(others)) return(data.table())
    y <- dd[fraction_label == ref][order(seed)]$recovered_traits
    if (length(y) < 2L) return(data.table())
    out <- rbindlist(lapply(others, function(f) {
        x <- dd[fraction_label == f][order(seed)]$recovered_traits
        if (length(x) < 2L) {
            return(data.table(fraction_label = f, n = length(x), n_ref = length(y),
                              delta = NA_real_, p = NA_real_))
        }
        data.table(fraction_label = f, n = length(x), n_ref = length(y),
                   delta = mean(y - x),
                   p = t.test(y, x, paired = TRUE, alternative = "greater")$p.value)
    }))
    out[, p_adj := p.adjust(p, method = "BH")]
    out[, stars := stars_for(p_adj)]
    out[]
}

make_ref_plot <- function(model_name) {
    d <- droplevels(traits[model == model_name])
    ref <- tail(levels(d$fraction_label), 1L)
    st <- ref_tests(d, ref)
    if (!nrow(d) || !nrow(st)) {
        message(sprintf("Not enough %s trait cells to compare against %s yet.", model_name, ref))
        return(invisible(NULL))
    }
    lv <- levels(d$fraction_label)
    span <- range(d$recovered_traits)
    yr <- diff(span)
    st <- copy(st)[!is.na(delta)][order(match(fraction_label, lv))]
    st[, `:=`(x = match(as.character(fraction_label), lv), xend = match(ref, lv))]
    st[, y := span[2] + yr * (0.04 + 0.052 * seq_len(.N))]
    fmt_p <- function(p) ifelse(p < 0.001, sprintf("%.1e", p), sprintf("%.3f", p))
    st[, label := sprintf("q = %s  (%+.0f)", fmt_p(p_adj), delta)]

    print(ggplot(d, aes(fraction_label, recovered_traits, fill = fraction_label)) +
        geom_boxplot(width = 0.6, outlier.shape = NA, alpha = 0.35, linewidth = 0.4, show.legend = FALSE) +
        geom_jitter(width = 0.12, size = 1.6, alpha = 0.75, show.legend = FALSE) +
        geom_segment(data = st, aes(x = x, xend = xend, y = y, yend = y), inherit.aes = FALSE, colour = "#444444", linewidth = 0.4) +
        geom_segment(data = st, aes(x = x, xend = x, y = y, yend = y - yr * 0.02), inherit.aes = FALSE, colour = "#444444", linewidth = 0.4) +
        geom_segment(data = st, aes(x = xend, xend = xend, y = y, yend = y - yr * 0.02), inherit.aes = FALSE, colour = "#444444", linewidth = 0.4) +
        geom_text(data = st, aes(x = (x + xend) / 2, y = y + yr * 0.016, label = label), inherit.aes = FALSE, size = 4, colour = "#222222") +
        scale_fill_viridis_d(end = 0.9, direction = -1) +
        scale_y_continuous(expand = expansion(mult = c(0.08, 0.06))) +
        labs(x = "Studies used", y = sprintf("Recovered traits (of %d tested)", unique(d$eligible_traits)),
             title = sprintf("%s - trait recovery at K = 1728 (FDR %.2f)", model_name, unique(d$fdr))) +
        theme_classic(base_size = 15) +
        theme(panel.grid.major.y = element_line(colour = "#E3E3E3", linewidth = 0.3),
              plot.title = element_text(face = "bold", size = 14)))

    ref_tests_by_model[[model_name]] <<- copy(st)[, model := model_name][]
}

ref_tests_by_model <- list()
options(repr.plot.width = 10, repr.plot.height = 7.5)
for (m in levels(traits$model)) make_ref_plot(m)

## Trait recovery by comparison

In [ ]:
rbindlist(ref_tests_by_model)[, .(
    model, comparison = sprintf("vs %s", fraction_label),
    n_ref, n, delta = round(delta, 1), p = signif(p, 3),
    p_adj = signif(p_adj, 3), stars
)]